# **LangGraph Reducers**

In LangGraph, a reducer tells the graph how to combine new output from a node with the existing State instead of overwriting it.

## **1. State Without Reducer**

In the example below, the second node overwrites what the first node appended to the `messages` key.

In [3]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage

class MyState(TypedDict):
    question: str
    messages: list[AnyMessage]

def add_system_message(state: MyState) -> dict:
    return {
        "messages": SystemMessage(content="You are a coding assistant")
    }

def add_user_question(state: MyState) -> dict:
    return {
        "messages": HumanMessage(content=state["question"])
    }

# Build and compile graph
graph = StateGraph(MyState)
graph.add_node("add_system_message", add_system_message)
graph.add_node("add_user_question", add_user_question)
graph.add_edge(START, "add_system_message")
graph.add_edge("add_system_message", "add_user_question")
graph.add_edge("add_user_question", END)
workflow = graph.compile()

# Invoke it
result = workflow.invoke(
    {"question": "What is a Python tuple?"}
)

print(result)


{'question': 'What is a Python tuple?', 'messages': HumanMessage(content='What is a Python tuple?', additional_kwargs={}, response_metadata={})}


## **2. State With a Reducer**

In the example below, we use the prebuilt `add_messages` reducer to append new messages to the list instead of overwriting it.

In [ ]:
from typing import TypedDict, Annotated # NEW
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages # NEW
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage

class MyState(TypedDict):
    question: str
    # Annotated allows to pass a reducer
    # The reducer defines how to handle merge conflicts
    # You can create your own reducer or use a pre-built one
    # add_messages appends the last messages to the previous ones
    messages: Annotated[list[AnyMessage], add_messages]

def add_system_message(state: MyState) -> dict:
    return {
        "messages": SystemMessage(content="You are a coding assistant")
    }

def add_user_question(state: MyState) -> dict:
    return {
        "messages": HumanMessage(content=state["question"])
    }

# Build and compile graph
graph = StateGraph(MyState)
graph.add_node("add_system_message", add_system_message)
graph.add_node("add_user_question", add_user_question)
graph.add_edge(START, "add_system_message")
graph.add_edge("add_system_message", "add_user_question")
graph.add_edge("add_user_question", END)
workflow = graph.compile()

# Invoke it
result = workflow.invoke(
    {"question": "What is a Python tuple?"}
)

print(result)

{'question': 'What is a Python tuple?', 'messages': [SystemMessage(content='You are a coding assistant', additional_kwargs={}, response_metadata={}, id='8e2c4f0c-9d68-4744-8236-f8a72e92cd6e'), HumanMessage(content='What is a Python tuple?', additional_kwargs={}, response_metadata={}, id='53a7a663-d306-4d8f-bca3-9180aac83033')]}


In [ ]:
# This is how the reducer works
add_messages(["msg 1", "msg 2"],"msg 3")

[HumanMessage(content='msg 1', additional_kwargs={}, response_metadata={}, id='d8a8eb75-0a91-42ae-9891-1c32422628eb'),
 HumanMessage(content='msg 2', additional_kwargs={}, response_metadata={}, id='46098628-6a9a-4cc2-babd-e8f88d094e08'),
 HumanMessage(content='msg 3', additional_kwargs={}, response_metadata={}, id='e057b33e-4b61-485a-9394-0bb86824888d')]

In [5]:
# This is a simplification of how the reducer works and how you can implement your custom reducer
def custom_add_messages(left: list[str], right: str | list[str]) -> list[str]:

  if not isinstance(right, list):
    right = [right]

  return left + right

custom_add_messages(["msg 1", "msg 2"],"msg 3")


['msg 1', 'msg 2', 'msg 3']

## **3. Use a Prebuilt State**

It is so popular for a LangGraph state to contain `messages: Annotated[list[AnyMessage], add_messages]` that a prebuilt state containing this pattern already exists: `MessagesState`

In [ ]:
from langgraph.graph import START, END, StateGraph, MessagesState # NEW
from langchain_core.messages import SystemMessage, HumanMessage

# NEW: we inherit from MessagesState
class MyState(MessagesState):
    question: str

def add_system_message(state: MyState) -> dict:
    return {
        "messages": SystemMessage(content="You are a coding assistant")
    }

def add_user_question(state: MyState) -> dict:
    return {
        "messages": HumanMessage(content=state["question"])
    }

# Build and compile graph
graph = StateGraph(MyState)
graph.add_node("add_system_message", add_system_message)
graph.add_node("add_user_question", add_user_question)
graph.add_edge(START, "add_system_message")
graph.add_edge("add_system_message", "add_user_question")
graph.add_edge("add_user_question", END)
workflow = graph.compile()

# Invoke it
result = workflow.invoke(
    {"question": "What is a Python tuple?"}
)

print(result)

{'messages': [SystemMessage(content='You are a coding assistant', additional_kwargs={}, response_metadata={}, id='63c8d220-1396-4cd1-b132-60393438ef39'), HumanMessage(content='What is a Python tuple?', additional_kwargs={}, response_metadata={}, id='552f2ab0-4fbc-48c2-8554-96af9056d5ad')], 'question': 'What is a Python tuple?'}
